# Qiskit Demo

This notebook demonstrates how to run quantum circuit simulations from within a Jupyter kernel
using the **agentic qiskit-node**.

1. **Bell state** (OpenQASM 2) — call the qiskit-node via the jupyter-node `/circuit/run` proxy
2. **GHZ state** (Qiskit Python) — build a 3-qubit circuit natively, display ASCII diagram
3. **Grover's algorithm** (2-qubit) — run with pubsub, stream results live
4. **LLM explanation** — ask the model-node to explain the results

> **Requirements**: run inside the agentic Docker network (`agentnet`)  
> The `SIBLING_QISKIT_NODE_URL` and `SIBLING_MODEL_NODE_URL` env vars are set automatically
> when using the orchestrator or jupyter-node compose stack.

## 0. Setup

In [ ]:
import os
import json
import base64
import time
from io import BytesIO

import httpx
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# jupyter-node API base — when running inside JupyterLab the node is on localhost.
JUPYTER_NODE_URL = os.environ.get("JUPYTER_NODE_URL", "http://localhost:8002")

# Qiskit-node — exposed by the jupyter-node /circuit/run proxy, but also available directly.
QISKIT_NODE_URL = os.environ.get("SIBLING_QISKIT_NODE_URL", "http://qiskit-node:8003")

# Model-node for LLM explanations.
MODEL_NODE_URL = os.environ.get("SIBLING_MODEL_NODE_URL", "http://model-node:8001")

print(f"Jupyter node : {JUPYTER_NODE_URL}")
print(f"Qiskit node  : {QISKIT_NODE_URL}")
print(f"Model node   : {MODEL_NODE_URL}")

# Quick health-check
for name, url in [("qiskit-node", QISKIT_NODE_URL), ("model-node", MODEL_NODE_URL)]:
    try:
        r = httpx.get(f"{url}/health", timeout=5)
        status = "OK" if r.status_code == 200 else f"HTTP {r.status_code}"
    except Exception as e:
        status = f"unreachable ({e})"
    print(f"  {name}: {status}")

## 1. Bell State (OpenQASM 2)

A Bell state is the simplest example of quantum entanglement.  
We send an OpenQASM 2 circuit to the qiskit-node via the jupyter-node `/circuit/run` proxy
and plot the measurement counts.

In [ ]:
BELL_QASM2 = """\
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
h q[0];
cx q[0],q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];
"""

resp = httpx.post(
    f"{JUPYTER_NODE_URL}/circuit/run",
    json={
        "code": BELL_QASM2,
        "language": "openqasm2",
        "shots": 1024,
        "include_diagram": True,
        "diagram_format": "text",
    },
    timeout=30,
)
resp.raise_for_status()
bell_result = resp.json()

print("Counts:", bell_result["counts"])
print("Status:", bell_result["status"])
print("\nCircuit diagram:")
print(bell_result.get("diagram", "(diagram not available)"))

In [ ]:
# Plot measurement histogram
counts = bell_result["counts"]
labels = sorted(counts.keys())
values = [counts[k] for k in labels]

plt.figure(figsize=(6, 4))
plt.bar(labels, values, color="steelblue", edgecolor="navy")
plt.xlabel("Bitstring")
plt.ylabel("Counts")
plt.title("Bell State |Φ⁺⟩ — 1024 shots")
plt.tight_layout()
plt.show()

In [ ]:
# Render the matplotlib circuit diagram (PNG)
resp_mpl = httpx.post(
    f"{JUPYTER_NODE_URL}/circuit/run",
    json={
        "code": BELL_QASM2,
        "language": "openqasm2",
        "shots": 1,
        "include_diagram": True,
        "diagram_format": "mpl",
    },
    timeout=30,
)
resp_mpl.raise_for_status()
png_b64 = resp_mpl.json().get("diagram", "")

if png_b64:
    png_bytes = base64.b64decode(png_b64)
    img = mpimg.imread(BytesIO(png_bytes))
    plt.figure(figsize=(5, 2))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Bell circuit (matplotlib renderer)")
    plt.tight_layout()
    plt.show()
else:
    print("No diagram returned.")

## 2. GHZ State (Qiskit Python, 3 qubits)

A Greenberger–Horne–Zeilinger (GHZ) state is a maximally entangled 3-qubit state.  
We build it using native Qiskit Python and post it to the proxy with `language='qiskit'`.

In [ ]:
GHZ_QISKIT = """
from qiskit import QuantumCircuit
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.cx(0, 2)
"""

resp = httpx.post(
    f"{JUPYTER_NODE_URL}/circuit/run",
    json={
        "code": GHZ_QISKIT,
        "language": "qiskit",
        "shots": 1024,
        "include_diagram": True,
        "diagram_format": "text",
    },
    timeout=30,
)
resp.raise_for_status()
ghz_result = resp.json()

print("GHZ counts:", ghz_result["counts"])
print("\nCircuit diagram:")
print(ghz_result.get("diagram", "(not available)"))

In [ ]:
# GHZ histogram — should show only '000' and '111'
counts = ghz_result["counts"]
labels = sorted(counts.keys())
values = [counts[k] for k in labels]

plt.figure(figsize=(6, 4))
plt.bar(labels, values, color="darkorchid", edgecolor="indigo")
plt.xlabel("Bitstring")
plt.ylabel("Counts")
plt.title("GHZ State |GHZ⟩ — 1024 shots")
plt.tight_layout()
plt.show()

## 3. Grover's Algorithm (2-qubit) with Pubsub Streaming

Grover's algorithm amplifies the probability of a marked state.  
We tag the cell with `# pubsub: qiskit-stream` so live IOPub output streams
to `ws://jupyter-node/ws/pubsub/qiskit-stream`.

We also pass `pubsub_topic` to the circuit run itself so the qiskit-node
publishes `running` / `completed` events to the same topic.

In [ ]:
# 2-qubit Grover — marks the |11⟩ state
GROVER_QASM2 = """\
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];

// Equal superposition
h q[0];
h q[1];

// Oracle: marks |11> with a phase flip (CZ)
cz q[0],q[1];

// Diffusion operator
h q[0];
h q[1];
x q[0];
x q[1];
cz q[0],q[1];
x q[0];
x q[1];
h q[0];
h q[1];

measure q[0] -> c[0];
measure q[1] -> c[1];
"""

In [ ]:
# pubsub: qiskit-stream
print("Submitting Grover circuit (pubsub topic: qiskit-stream) ...")

resp = httpx.post(
    f"{JUPYTER_NODE_URL}/circuit/run",
    json={
        "code": GROVER_QASM2,
        "language": "openqasm2",
        "shots": 1024,
        "pubsub_topic": "qiskit-stream",
        "include_diagram": True,
        "diagram_format": "text",
    },
    timeout=30,
)
resp.raise_for_status()
grover_result = resp.json()

print("Grover counts:", grover_result["counts"])
print("\nCircuit diagram:")
print(grover_result.get("diagram", "(not available)"))

In [ ]:
# Grover histogram — '11' should dominate (~97% probability in ideal simulation)
counts = grover_result["counts"]
labels = sorted(counts.keys())
values = [counts[k] for k in labels]
total = sum(values)

plt.figure(figsize=(6, 4))
bars = plt.bar(labels, values, color="tomato", edgecolor="darkred")
for bar, v in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
             f"{v/total:.0%}", ha="center", va="bottom", fontsize=10)
plt.xlabel("Bitstring")
plt.ylabel("Counts")
plt.title("Grover's Algorithm (marks |11⟩) — 1024 shots")
plt.tight_layout()
plt.show()

## 4. LLM Explanation

Call the model-node via the jupyter-node `/infer` proxy to generate a plain-language
explanation of the Grover simulation results.

> **Requires**: model-node running with a loaded LLM (e.g. `phi3:mini`).

In [ ]:
prompt = (
    f"I ran Grover's algorithm on a 2-qubit system targeting the |11> state. "
    f"The simulation used 1024 shots and produced these measurement counts: {grover_result['counts']}. "
    "Please explain the result in plain language, including why the marked state dominates "
    "and what this demonstrates about quantum amplitude amplification."
)

# In production the model call can take longer than 2 minutes; retry on read timeouts.
timeouts = [180, 300]
last_err = None
for t in timeouts:
    try:
        resp = httpx.post(
            f"{JUPYTER_NODE_URL}/infer",
            json={"prompt": prompt},
            timeout=t,
        )
        resp.raise_for_status()
        explanation = resp.json().get("text", resp.text)
        print(explanation)
        break
    except httpx.ReadTimeout as exc:
        last_err = exc
        print(f"Model response timed out after {t}s; retrying...")
else:
    print("Model explanation is taking longer than expected.")
    print("Tip: rerun this cell, or check model-node health and loaded model.")
    print(f"Last error: {type(last_err).__name__}: {last_err}")

## 5. Summary

| Circuit | Qubits | Language | Key result |
|---------|--------|----------|------------|
| Bell state | 2 | OpenQASM 2 | ~50% `00`, ~50% `11` — entanglement |
| GHZ state | 3 | Qiskit Python | ~50% `000`, ~50% `111` — tripartite entanglement |
| Grover's algorithm | 2 | OpenQASM 2 | `11` dominates (~97%) — amplitude amplification |

All circuits were simulated by the **qiskit-node** (Qiskit Aer statevector simulator)
via the **jupyter-node** `/circuit/run` proxy route.